In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys

sys.path.append("../src")

from sentence_transformers import SentenceTransformer
from data_loader import load_candidates
from job_representation import build_job_description
from retrieval import Retriever
from ranking import Ranker

# Load model only once
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

retriever = Retriever(embedding_model)
ranker = Ranker(embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:

from data_loader import load_candidates

candidates = load_candidates(
    "../data/candidates.jsonl",
    limit=100
)

print(len(candidates))


100


In [5]:
from docx import Document
from job_representation import build_job_description

doc = Document("../India_Runs_Hackathon/job_description.docx")

jd_text = "\n".join(
    para.text
    for para in doc.paragraphs
)

job = build_job_description(jd_text)

In [6]:
ranker.build_requirement_embeddings(job)

ranker.build_experience_embeddings(candidates)

In [7]:
from explanation import ExplanationGenerator

In [8]:
rankings = ranker.rank_candidates(
    job,
    candidates
)

In [8]:
print(rankings[0].keys())

dict_keys(['candidate', 'candidate_id', 'final_score', 'evidence_score', 'skill_score', 'experience_score', 'behavior_score'])


In [9]:
print(rankings[0]["candidate"])

Candidate(candidate_id='CAND_0000031', retrieval_document="Summary: Machine learning engineer with 6.0 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I've learned that most retrieval problems are actually evaluation problems in disguise. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I'd own a meaningful piece of the ML stack.\nHeadline: Recommendation Systems Engineer | Search, Ranking & Retrieval\nSkills: Go, MLflow, FAISS, Pinecone, Angular, Image Classification, Machine Learning, Speech Recognition, BentoML, MLOps, Embeddings, Information Retr

In [1]:
explainer = ExplanationGenerator()

NameError: name 'ExplanationGenerator' is not defined

In [16]:
#testing exp generator
from submission import generate_submission

submission = generate_submission(
    job,
    candidates,
    ranker,
    explainer
)

submission.head()

,candidate_id,rank,reasoning
0,CAND_0000031,1,Production experience in recommendation system...
1,CAND_0000082,2,Relevant production engineering experience. Ke...
2,CAND_0000063,3,Relevant production engineering experience. Ke...
3,CAND_0000016,4,Relevant production engineering experience. Ke...
4,CAND_0000001,5,Relevant production engineering experience. Ke...


In [13]:
rankings = ranker.rank_candidates(
    job,
    candidates
)

print(len(rankings))

100


In [14]:
top_candidates = rankings[:100]

print(len(top_candidates))

100


In [15]:
print(len(rows))

NameError: name 'rows' is not defined

In [11]:
candidate = next(
    c for c in candidates
    if c.candidate_id == "CAND_0000001"
)

ranking = next(
    r for r in rankings
    if r["candidate_id"] == "CAND_0000001"
)

print(
    explainer.generate_reasoning(
        job,
        candidate,
        ranking
    )
)

Relevant production engineering experience. 6.9 years of experience matches the preferred range. Behavioral signals indicate actively seeking opportunities.


In [12]:
from submission import generate_submission

submission = generate_submission(
    job,
    candidates,
    ranker,
    explainer
)

submission.head()

,candidate_id,score,reasoning
0,CAND_0000031,0.4782,Production experience in recommendation system...


In [6]:
#score exp
for cid in [
    "CAND_0000031",
    "CAND_0000082",
    "CAND_0000001",
    "CAND_0000074"
]:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    print(
        cid,
        candidate.metadata["years_of_experience"],
        ranker._score_experience(job, candidate)
    )

CAND_0000031 6.0 1.0
CAND_0000082 7.4 1.0
CAND_0000001 6.9 1.0
CAND_0000074 1.9 0.38


In [7]:
candidate = next(
    c for c in candidates
    if c.candidate_id == "CAND_0000031"
)

print(candidate.signals)

{'profile_completeness_score': 83.4, 'signup_date': '2026-01-28', 'last_active_date': '2026-05-24', 'open_to_work_flag': True, 'profile_views_received_30d': 194, 'applications_submitted_30d': 2, 'recruiter_response_rate': 0.91, 'avg_response_time_hours': 76.1, 'skill_assessment_scores': {'MLflow': 75.1, 'FAISS': 68.4, 'Pinecone': 53.6, 'Image Classification': 57.1}, 'connection_count': 832, 'endorsements_received': 177, 'notice_period_days': 60, 'expected_salary_range_inr_lpa': {'min': 27.3, 'max': 60.2}, 'preferred_work_mode': 'flexible', 'willing_to_relocate': True, 'github_activity_score': 32.6, 'search_appearance_30d': 778, 'saved_by_recruiters_30d': 13, 'interview_completion_rate': 0.6, 'offer_acceptance_rate': 0.38, 'verified_email': False, 'verified_phone': True, 'linkedin_connected': False}


In [8]:
#score signals tets 
for cid in [
    "CAND_0000031",
    "CAND_0000082",
    "CAND_0000001",
    "CAND_0000074"
]:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    print(
        cid,
        round(
            ranker._score_behavior(candidate),
            3
        )
    )

CAND_0000031 0.793
CAND_0000082 0.641
CAND_0000001 0.591
CAND_0000074 0.379


In [13]:
results = ranker.rank_candidates(
    job,
    candidates
)

results[:10]

[{'candidate_id': 'CAND_0000031',
  'final_score': 0.4781632833373547,
  'evidence_score': 0.30928656667470933,
  'skill_score': 0.1,
  'experience_score': 1.0,
  'behavior_score': 0.7926},
 {'candidate_id': 'CAND_0000082',
  'final_score': 0.3926870748651028,
  'evidence_score': 0.18113414973020553,
  'skill_score': 0.16,
  'experience_score': 1.0,
  'behavior_score': 0.6406000000000001},
 {'candidate_id': 'CAND_0000063',
  'final_score': 0.3593620457775891,
  'evidence_score': 0.12292409155517817,
  'skill_score': 0.0,
  'experience_score': 1.0,
  'behavior_score': 0.7395},
 {'candidate_id': 'CAND_0000016',
  'final_score': 0.35622325835585594,
  'evidence_score': 0.13242651671171188,
  'skill_score': 0.0,
  'experience_score': 1.0,
  'behavior_score': 0.7000500000000001},
 {'candidate_id': 'CAND_0000001',
  'final_score': 0.34599858152270313,
  'evidence_score': 0.15569716304540634,
  'skill_score': 0.0,
  'experience_score': 1.0,
  'behavior_score': 0.59075},
 {'candidate_id': 'CAN

In [14]:
results = ranker.rank_candidates(job, candidates)

print(results[:10])

[{'candidate_id': 'CAND_0000031', 'final_score': 0.4781632833373547, 'evidence_score': 0.30928656667470933, 'skill_score': 0.1, 'experience_score': 1.0, 'behavior_score': 0.7926}, {'candidate_id': 'CAND_0000082', 'final_score': 0.3926870748651028, 'evidence_score': 0.18113414973020553, 'skill_score': 0.16, 'experience_score': 1.0, 'behavior_score': 0.6406000000000001}, {'candidate_id': 'CAND_0000063', 'final_score': 0.3593620457775891, 'evidence_score': 0.12292409155517817, 'skill_score': 0.0, 'experience_score': 1.0, 'behavior_score': 0.7395}, {'candidate_id': 'CAND_0000016', 'final_score': 0.35622325835585594, 'evidence_score': 0.13242651671171188, 'skill_score': 0.0, 'experience_score': 1.0, 'behavior_score': 0.7000500000000001}, {'candidate_id': 'CAND_0000001', 'final_score': 0.34599858152270313, 'evidence_score': 0.15569716304540634, 'skill_score': 0.0, 'experience_score': 1.0, 'behavior_score': 0.59075}, {'candidate_id': 'CAND_0000093', 'final_score': 0.3401156597036123, 'evidenc

In [ ]:
import importlib
import ranking

importlib.reload(ranking)

from ranking import Ranker

In [19]:
ranker = Ranker(embedding_model)

In [6]:
#score skill test
candidate = next(
    c for c in candidates
    if c.candidate_id == "CAND_0000031"
)

print(
    ranker._score_skills(
        job,
        candidate
    )
)

0.1


In [7]:
for cid in [
    "CAND_0000031",
    "CAND_0000082",
    "CAND_0000001",
    "CAND_0000074"
]:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    print(
        cid,
        round(
            ranker._score_skills(job, candidate),
            3
        )
    )

CAND_0000031 0.1
CAND_0000082 0.16
CAND_0000001 0.0
CAND_0000074 0.28


In [6]:
#metadata test
candidate = candidates[0]

print(candidate.metadata)

{'num_skills': 17, 'num_jobs': 2, 'num_education': 1, 'years_of_experience': 6.9}


In [ ]:
#score evidence test
andidate = next(
    c for c in candidates
    if c.candidate_id == "CAND_0000031"
)

score = ranker._score_evidence(
    job,
    candidate
)

print(score)

retrieval systems         Weight=3.0 Best=0.000
retrieval systems         Weight=3.0 Best=0.339
retrieval systems         Weight=3.0 Best=0.369
retrieval systems         Weight=3.0 Best=0.369
1.1072203516960144
3.0
ranking systems           Weight=3.0 Best=0.000
ranking systems           Weight=3.0 Best=0.448
ranking systems           Weight=3.0 Best=0.448
ranking systems           Weight=3.0 Best=0.448
2.4500079452991486
6.0
recommendation systems    Weight=3.0 Best=0.000
recommendation systems    Weight=3.0 Best=0.501
recommendation systems    Weight=3.0 Best=0.501
recommendation systems    Weight=3.0 Best=0.501
3.953246980905533
9.0
search systems            Weight=3.0 Best=0.000
search systems            Weight=3.0 Best=0.331
search systems            Weight=3.0 Best=0.417
search systems            Weight=3.0 Best=0.417
5.204311966896057
12.0
embeddings                Weight=2.5 Best=0.000
embeddings                Weight=2.5 Best=0.101
embeddings                Weight=2.5 Best=0.1

In [7]:
for cid in [
    "CAND_0000031",
    "CAND_0000001",
    "CAND_0000047",
    "CAND_0000050"
]:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    score = ranker._score_evidence(job, candidate)

    print(cid, score)

CAND_0000031 0.30928656667470933
CAND_0000001 0.15569716304540634
CAND_0000047 0.10386441245675088
CAND_0000050 0.13281193122267723


In [7]:
test_candidates = [
    "CAND_0000031",
    "CAND_0000074",
    "CAND_0000001",
    "CAND_0000055",
    "CAND_0000082"
]

for cid in test_candidates:

    candidate = next(
        c for c in candidates
        if c.candidate_id == cid
    )

    score = ranker._score_evidence(
        job,
        candidate
    )

    print(f"{cid}: {score:.4f}")

retrieval systems         Weight=3.0 Best=0.000
retrieval systems         Weight=3.0 Best=0.339
retrieval systems         Weight=3.0 Best=0.369
retrieval systems         Weight=3.0 Best=0.369
1.1072203516960144
3.0
ranking systems           Weight=3.0 Best=0.000
ranking systems           Weight=3.0 Best=0.448
ranking systems           Weight=3.0 Best=0.448
ranking systems           Weight=3.0 Best=0.448
2.4500079452991486
6.0
recommendation systems    Weight=3.0 Best=0.000
recommendation systems    Weight=3.0 Best=0.501
recommendation systems    Weight=3.0 Best=0.501
recommendation systems    Weight=3.0 Best=0.501
3.953246980905533
9.0
search systems            Weight=3.0 Best=0.000
search systems            Weight=3.0 Best=0.331
search systems            Weight=3.0 Best=0.417
search systems            Weight=3.0 Best=0.417
5.204311966896057
12.0
embeddings                Weight=2.5 Best=0.000
embeddings                Weight=2.5 Best=0.101
embeddings                Weight=2.5 Best=0.1